In [2]:
'''
gets the summary of the old labeled dataset
'''


import pandas as pd

FILE_PATH = "labeled_dataset_new_deduped.xlsx"   # <-- change this to your file's path

df = pd.read_excel(FILE_PATH, dtype={"id": str})

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}\n")

label_counts = df["Label"].value_counts(dropna=False).sort_index()
print("Label counts:")
for label, count in label_counts.items():
    print(f"  Label {label}: {count}")

print(f"\nLabel 0: {label_counts.get(0, 0)}")
print(f"Label 1: {label_counts.get(1, 0)}")

print("\nSimilarity score stats:")
print(df["similarity_score"].describe())

summary = pd.DataFrame({
    "Label": label_counts.index,
    "Count": label_counts.values,
    "Percentage": (label_counts.values / len(df) * 100).round(2)
})
# summary.to_excel("label_summary.xlsx", index=False)
# print("\nSaved summary to label_summary.xlsx")

Total rows: 406
Columns: ['id', 'text', 'similarity_score', 'Label', 'label_reason']

Label counts:
  Label 0: 208
  Label 1: 198

Label 0: 208
Label 1: 198

Similarity score stats:
count    289.000000
mean       0.315999
std        0.065424
min        0.250353
25%        0.269924
50%        0.293415
75%        0.335667
max        0.617340
Name: similarity_score, dtype: float64


In [12]:
'''
randomly sampled from the master posts randomly 400  posts that gets labeled by claude 
change to do: min char posts with more than 20 char 

'''

import pandas as pd

Full_Data = "master_posts.csv"
Labeled_Data = "labeled_dataset.xlsx"

# load the data
full_df = pd.read_csv(Full_Data)
labeled_df = pd.read_excel(Labeled_Data)

print("=== Full dataset columns ===")
print(full_df.columns.tolist())

print("\n=== Labeled dataset columns ===")
print(labeled_df.columns.tolist())

# Remove posts that have already been labeled
remaining = full_df[~full_df["id"].isin(labeled_df["id"])].copy()

print(f"Remaining unlabeled posts: {len(remaining)}")
# Filter out posts that are too short (min 20 characters)
MIN_CHARS = 20
remaining = remaining[remaining["text"].astype(str).str.len() > MIN_CHARS].copy()
print(f"Remaining after min-char filter (> {MIN_CHARS} chars): {len(remaining)}")
# Randomly sample 300 posts
sample = remaining.sample(n=800, random_state=42).copy()

# Keep only the columns needed for annotation
sample = sample[["id", "text"]]

# Add empty annotation columns
sample["similarity_score"] = ""
sample["Label"] = ""
sample["label_reason"] = ""

# Save as CSV
sample.to_csv("annotation_sample_800.csv", index=False)

print(f"Saved {len(sample)} posts to annotation_sample_800.csv")

=== Full dataset columns ===
['id', 'subreddit', 'author', 'created_date', 'created_utc', 'year', 'month', 'day_of_week', 'hour', 'year_month', 'title', 'selftext', 'text', 'cleaned_text', 'original_text_len', 'cleaned_text_len', 'word_count', 'score', 'upvote_ratio', 'num_comments', 'link_flair_text', 'permalink', 'url']

=== Labeled dataset columns ===
['id', 'text', 'similarity_score', 'Label', 'label_reason']
Remaining unlabeled posts: 354668
Remaining after min-char filter (> 20 chars): 354668
Saved 800 posts to annotation_sample_800.csv


In [13]:
"""
Now pass that annotation sample to claude to put label -> save the file then run this for a check ->
annotation_sample_300_labeled.csv
first check the label distribution
then we want to add the label 1 samples in the labeled_dataset.xlsx sheet 
"""

import pandas as pd

FILE_PATH = "annotation_sample_800_labeled.csv"   # <-- change this to your file's path

df = pd.read_csv(FILE_PATH, dtype={"id": str})

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}\n")

label_counts = df["Label"].value_counts(dropna=False).sort_index()
print("Label counts:")
for label, count in label_counts.items():
    print(f"  Label {label}: {count}")

print(f"\nLabel 0: {label_counts.get(0, 0)}")
print(f"Label 1: {label_counts.get(1, 0)}")

print("\nSimilarity score stats:")
print(df["similarity_score"].describe())

summary = pd.DataFrame({
    "Label": label_counts.index,
    "Count": label_counts.values,
    "Percentage": (label_counts.values / len(df) * 100).round(2)
})
# summary.to_excel("label_summary.xlsx", index=False)
# print("\nSaved summary to label_summary.xlsx")

Total rows: 800
Columns: ['id', 'text', 'similarity_score', 'Label', 'label_reason']

Label counts:
  Label 0: 725
  Label 1: 75

Label 0: 725
Label 1: 75

Similarity score stats:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: similarity_score, dtype: float64


Now just add the 52 samples where label = 1 rows in annotation_sample_300_labeled.csv to the old dataset labeled_dataset.xlsx and name it labeled_dataset_new.xlsx

In [16]:
"""
Appends the Label = 1 rows from annotation_sample_300_labeled.csv onto the
'Labeled Dataset' sheet of labeled_dataset.xlsx, updates the Summary sheet,
and saves the result as labeled_dataset_new.xlsx.

Usage:
    python3 add_label1_to_dataset.py
"""
import pandas as pd
from openpyxl import load_workbook

NEW_SAMPLE = "annotation_sample_800_labeled.csv"
MAIN_DATASET = "labeled_dataset.xlsx"  # <- point this at your ORIGINAL 309-row file
OUTPUT_DATASET = "labeled_dataset_new.xlsx"
SHEET_NAME = "Labeled Dataset"
COLUMNS = ["id", "text", "similarity_score", "Label", "label_reason"]

# ---------- Load ----------
new_df = pd.read_csv(NEW_SAMPLE)
main_df = pd.read_excel(MAIN_DATASET, sheet_name=SHEET_NAME)

# ---------- Filter to Label == 1 rows only ----------
new_label1 = new_df[new_df["Label"] == 1][COLUMNS].copy()
print(f"New Label=1 rows to add: {len(new_label1)}")

new_label2 = new_df[new_df["Label"] == 0].sample(n=26, random_state=42)
print(f"New Label=0 rows to add: {len(new_label2)}")
# ---------- Check for id overlap with existing dataset ----------
overlap = set(new_label1["id"]) & set(new_label2["id"]) & set(main_df["id"])
if overlap:
    print(f"WARNING: {len(overlap)} id(s) already exist in the main dataset "
          f"and will be skipped: {overlap}")
    new_label1 = new_label1[~new_label1["id"].isin(overlap)]
    new_label2 = new_label2[~new_label2["id"].isin(overlap)]



# ---------- Append rows into the workbook (preserves formatting) ----------
wb = load_workbook(MAIN_DATASET)
ws = wb[SHEET_NAME]

for _, row in new_label1.iterrows():
    ws.append([row["id"], row["text"], row["similarity_score"],
               row["Label"], row["label_reason"]])
    
for _, row in new_label2.iterrows():
    ws.append([row["id"], row["text"], row["similarity_score"],
               row["Label"], row["label_reason"]])

total = ws.max_row - 1  # minus header
print(f"Labeled Dataset sheet: {total} total rows after append")

# ---------- Update Summary sheet ----------
if "Summary" in wb.sheetnames:
    summary_ws = wb["Summary"]
    label_col_idx = COLUMNS.index("Label") + 1
    labels = [ws.cell(row=r, column=label_col_idx).value for r in range(2, ws.max_row + 1)]
    n1 = sum(1 for l in labels if l == 1)
    n0 = sum(1 for l in labels if l == 0)

    for r in range(2, summary_ws.max_row + 1):
        metric = summary_ws.cell(row=r, column=1).value
        if metric == "Total Posts":
            summary_ws.cell(row=r, column=2).value = total
        elif metric == "In-Scope (Label = 1)":
            summary_ws.cell(row=r, column=2).value = n1
        elif metric == "Out-of-Scope (Label = 0)":
            summary_ws.cell(row=r, column=2).value = n0

    print(f"Summary updated -> total: {total}, label=1: {n1}, label=0: {n0}")

# ---------- Save ----------
wb.save(OUTPUT_DATASET)
print(f"Saved: {OUTPUT_DATASET}")

New Label=1 rows to add: 75
New Label=0 rows to add: 26
Labeled Dataset sheet: 410 total rows after append
Summary updated -> total: 410, label=1: 200, label=0: 210
Saved: labeled_dataset_new.xlsx


In [17]:

import pandas as pd

FILE_PATH = "labeled_dataset_new.xlsx"   # <-- change this to your file's path

df = pd.read_excel(FILE_PATH, dtype={"id": str})

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}\n")

label_counts = df["Label"].value_counts(dropna=False).sort_index()
print("Label counts:")
for label, count in label_counts.items():
    print(f"  Label {label}: {count}")

print(f"\nLabel 0: {label_counts.get(0, 0)}")
print(f"Label 1: {label_counts.get(1, 0)}")

print("\nSimilarity score stats:")
print(df["similarity_score"].describe())

summary = pd.DataFrame({
    "Label": label_counts.index,
    "Count": label_counts.values,
    "Percentage": (label_counts.values / len(df) * 100).round(2)
})
summary.to_excel("label_summary.xlsx", index=False)
print("\nSaved summary to label_summary.xlsx")

Total rows: 410
Columns: ['id', 'text', 'similarity_score', 'Label', 'label_reason']

Label counts:
  Label 0: 210
  Label 1: 200

Label 0: 210
Label 1: 200

Similarity score stats:
count    289.000000
mean       0.315999
std        0.065424
min        0.250353
25%        0.269924
50%        0.293415
75%        0.335667
max        0.617340
Name: similarity_score, dtype: float64

Saved summary to label_summary.xlsx


In [26]:
"""
Checks labeled_dataset_new.xlsx for duplicate ids, reports them, and saves
a deduped copy.

Dedup rule: when an id appears more than once, keep the row that has a
non-null similarity_score (earlier investigation found dup rows share the
same Label/label_reason but differ only in whether similarity_score was
filled in - looks like two merged export batches). If multiple copies
still tie (e.g. both have/lack similarity_score), keep the first.

NOTE: this does not touch the original file in /mnt/user-data/uploads/
(read-only, and raw files should never be modified in place). It writes
a new file: labeled_dataset_new_deduped.xlsx
"""
import pandas as pd

SRC = "labeled_dataset_new.xlsx"
OUT = "labeled_dataset_new_deduped.xlsx"

df = pd.read_excel(SRC)
print(f"Loaded {len(df)} rows")

dup_mask = df.duplicated(subset="id", keep=False)
n_dupes = dup_mask.sum()
print(f"Found {n_dupes} rows involved in duplicate ids ({df['id'].duplicated().sum()} extra copies to drop)")

if n_dupes > 0:
    print("\nDuplicate rows:")
    print(df[dup_mask].sort_values("id")[["id", "Label", "similarity_score", "label_reason"]].to_string())

# Keep the row with a non-null similarity_score when there's a choice
df["_has_sim"] = df["similarity_score"].notna()
deduped = (
    df.sort_values("_has_sim", ascending=False)
      .drop_duplicates(subset="id", keep="first")
      .drop(columns="_has_sim")
      .sort_index()
      .reset_index(drop=True)
)

print(f"\nAfter dedup: {len(deduped)} rows (removed {len(df) - len(deduped)})")

deduped.to_excel(OUT, index=False)
print(f"Saved: {OUT}")

Loaded 410 rows
Found 8 rows involved in duplicate ids (4 extra copies to drop)

Duplicate rows:
          id  Label  similarity_score                                            label_reason
44   1aso6ah      0          0.290671                                          Matched: sleep
300  1aso6ah      0               NaN                                          Matched: sleep
114  1bwwk0b      0          0.281315                               Matched: sleeper, layoffs
305  1bwwk0b      0               NaN                               Matched: sleeper, layoffs
47   1dcfy3b      1          0.269627            Matched: pressure, sleep, tired, can't sleep
302  1dcfy3b      1               NaN            Matched: pressure, sleep, tired, can't sleep
36   1dtiput      1          0.299787  Matched: burn out, sleep, motivation to grow and learn
290  1dtiput      1               NaN  Matched: burn out, sleep, motivation to grow and learn

After dedup: 406 rows (removed 4)
Saved: labeled_dataset

In [25]:
# July 9 
import pandas as pd
 
INPUT_PATH  = "/Users/nadia/Desktop/redditRun_june/master_posts_classified.csv"
OUTPUT_PATH = "/Users/nadia/Desktop/redditRun_june/master_posts_burnout_only.csv"
LABEL_COL = "predicted_label"
 
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows from {INPUT_PATH}")
print(f"Columns: {list(df.columns)}\n")
print(f"Label distribution:\n{df[LABEL_COL].value_counts()}\n")
filtered = df[df[LABEL_COL] == 1].copy()
print(f"Rows with {LABEL_COL} == 1: {len(filtered)}")
 
filtered.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
 

Loaded 366115 rows from /Users/nadia/Desktop/redditRun_june/master_posts_classified.csv
Columns: ['id', 'subreddit', 'author', 'created_date', 'created_utc', 'year', 'month', 'day_of_week', 'hour', 'year_month', 'title', 'selftext', 'text', 'cleaned_text', 'original_text_len', 'cleaned_text_len', 'word_count', 'score', 'upvote_ratio', 'num_comments', 'link_flair_text', 'permalink', 'url', 'text_processed', 'predicted_label', 'prediction_confidence']

Label distribution:
predicted_label
0.0    210206
1.0    144652
Name: count, dtype: int64

Rows with predicted_label == 1: 144652
Saved: /Users/nadia/Desktop/redditRun_june/master_posts_burnout_only.csv


In [3]:
import pandas as pd

classified = pd.read_csv("master_posts_classified.csv")
labeled = pd.read_excel("labeled_dataset_new.xlsx")  # or _deduped, whichever is final

before = len(classified)
classified = classified[~classified["id"].isin(labeled["id"])]
print(f"Dropped {before - len(classified)} rows; {len(classified)} remain")

classified.to_csv("master_posts_classified_final.csv", index=False)

Dropped 294 rows; 365821 remain
